# Deney 1 Frame Seçimi — Güncel MediaPipe Tasks API

Bu notebook:

- Google Drive'ı bağlar.
- Daha önce çıkarılmış `real_frames` ve `fake_frames` görüntülerini okur.
- Yüz bulunan görüntüleri MediaPipe **Tasks Face Detector** ile seçer.
- Genel toplam **3000 frame** üretir:
  - **1500 Real**
  - **1500 Fake**
- Her sınıfı kendi içinde:
  - `%80 train`
  - `%10 validation`
  - `%10 test`
  olarak düzenler.
- Kaynak `train/val/test` ayrımını korur.
- Kaynak klasörlerden hiçbir dosya silmez veya taşımaz.
- Yalnızca `OUTPUT_ROOT` içindeki `Real` ve `Fake` çıktı klasörlerini temizleyip yeniden oluşturur.
- Metadata CSV ve yüz bulunamayan görüntüler için log üretir.

> Bu sürüm eski `mp.solutions` arayüzünü kullanmaz. Güncel `mp.tasks.vision.FaceDetector` arayüzünü kullanır.


## 1. Kurulum

Aşağıdaki hücre resmi MediaPipe paketini kurar. `--no-deps` kullanıldığı için Colab'daki NumPy, TensorFlow, OpenCV ve protobuf sürümlerini değiştirmez.

Hücreyi çalıştırdıktan sonra MediaPipe daha önce farklı bir sürümle yüklenmişse:

**Çalışma zamanı → Oturumu yeniden başlat**

seçeneğini bir kez kullan ve notebook'u baştan çalıştır.


In [2]:
# Colab ortamında MediaPipe yoksa kur; varsa tekrar kaldırıp paketleri bozma.
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("mediapipe") is None:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install",
        "--quiet", "--no-deps", "mediapipe==0.10.35"
    ])
    print("MediaPipe kuruldu. Bu hücreden sonra çalışma zamanını bir kez yeniden başlat.")
else:
    import mediapipe as _mp
    print("MediaPipe zaten kurulu:", _mp.__version__)


MediaPipe kuruldu. Bu hücreden sonra çalışma zamanını bir kez yeniden başlat.


## 2. Drive Bağlantısı ve Kütüphaneler


In [3]:
from google.colab import drive
drive.mount("/content/drive")


Mounted at /content/drive


In [4]:
import os
import csv
import cv2
import json
import shutil
import random
import urllib.request
from pathlib import Path

import numpy as np
import mediapipe as mp
from tqdm.auto import tqdm

print("MediaPipe sürümü:", mp.__version__)
print("OpenCV sürümü:", cv2.__version__)
print("NumPy sürümü:", np.__version__)
print("MediaPipe Tasks mevcut mu:", hasattr(mp, "tasks"))

if not hasattr(mp, "tasks"):
    raise RuntimeError(
        "MediaPipe Tasks API yüklenmemiş görünüyor. "
        "Çalışma zamanını yeniden başlatıp notebook'u baştan çalıştır."
    )


MediaPipe sürümü: 0.10.35
OpenCV sürümü: 4.13.0
NumPy sürümü: 2.0.2
MediaPipe Tasks mevcut mu: True


## 3. Ayarlar ve Klasör Yolları

Aşağıdaki üç yolu kendi Drive klasör yapına göre kontrol et.

Kaynak klasörler sadece okunur:

- `REAL_FRAMES_ROOT`
- `FAKE_FRAMES_ROOT`

Silme işlemi yalnızca `OUTPUT_ROOT/Real` ve `OUTPUT_ROOT/Fake` klasörlerine uygulanır.


In [5]:
# ============================================================
# DRIVE YOLLARI — KENDİ KLASÖR ADINA GÖRE KONTROL ET
# ============================================================

PROJECT_ROOT = "/content/drive/MyDrive/AISC DeepFake Çalışmaları/FaceForensics_Projesi"

REAL_FRAMES_ROOT = os.path.join(PROJECT_ROOT, "real_frames")
FAKE_FRAMES_ROOT = os.path.join(PROJECT_ROOT, "fake_frames")

OUTPUT_ROOT = "/content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Deney 1 Frame"

# ============================================================
# HEDEF SAYILAR
# ============================================================

TOTAL_TARGET = 3000
TARGET_PER_CLASS = TOTAL_TARGET // 2  # 1500 Real + 1500 Fake

TRAIN_RATIO = 0.80
VAL_RATIO = 0.10
TEST_RATIO = 0.10

TARGETS_PER_SPLIT = {
    "train": 1200,
    "val": 150,
    "test": 150,
}

FACE_MIN_CONFIDENCE = 0.50
RANDOM_SEED = 42
IMG_EXTENSIONS = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

METADATA_CSV_PATH = os.path.join(OUTPUT_ROOT, "secim_metadata.csv")
NO_FACE_LOG_PATH = os.path.join(OUTPUT_ROOT, "yuzsuz_veya_okunamayan_frameler.csv")
CHECKPOINT_JSON_PATH = os.path.join(OUTPUT_ROOT, "islem_checkpoint.json")
RUN_SUMMARY_PATH = os.path.join(OUTPUT_ROOT, "islem_ozeti.json")

# MediaPipe Tasks Face Detector model dosyası
MODEL_PATH = "/content/blaze_face_short_range.tflite"
MODEL_URL = (
    "https://storage.googleapis.com/mediapipe-models/"
    "face_detector/blaze_face_short_range/float16/latest/"
    "blaze_face_short_range.tflite"
)

# Sayısal doğrulamalar
assert TOTAL_TARGET == 3000
assert TARGET_PER_CLASS == 1500
assert sum(TARGETS_PER_SPLIT.values()) == TARGET_PER_CLASS
assert abs(TRAIN_RATIO + VAL_RATIO + TEST_RATIO - 1.0) < 1e-9

print("Real kaynak:", REAL_FRAMES_ROOT)
print("Fake kaynak:", FAKE_FRAMES_ROOT)
print("Çıktı:", OUTPUT_ROOT)
print("Split hedefleri:", TARGETS_PER_SPLIT)


Real kaynak: /content/drive/MyDrive/AISC DeepFake Çalışmaları/FaceForensics_Projesi/real_frames
Fake kaynak: /content/drive/MyDrive/AISC DeepFake Çalışmaları/FaceForensics_Projesi/fake_frames
Çıktı: /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Deney 1 Frame
Split hedefleri: {'train': 1200, 'val': 150, 'test': 150}


## 4. Güvenlik ve Kaynak Klasör Kontrolleri


In [6]:
def normalize_path(path):
    return os.path.realpath(os.path.abspath(path))

real_root_norm = normalize_path(REAL_FRAMES_ROOT)
fake_root_norm = normalize_path(FAKE_FRAMES_ROOT)
output_root_norm = normalize_path(OUTPUT_ROOT)

# Çıktı yolu kaynak klasörlerle aynı veya onların içinde olamaz.
assert output_root_norm != real_root_norm
assert output_root_norm != fake_root_norm
assert not output_root_norm.startswith(real_root_norm + os.sep)
assert not output_root_norm.startswith(fake_root_norm + os.sep)

for root in [REAL_FRAMES_ROOT, FAKE_FRAMES_ROOT]:
    if not os.path.isdir(root):
        raise FileNotFoundError(f"Kaynak klasör bulunamadı: {root}")

    for split in ["train", "val", "test"]:
        split_path = os.path.join(root, split)
        if not os.path.isdir(split_path):
            raise FileNotFoundError(f"Kaynak split klasörü bulunamadı: {split_path}")

print("Kaynak klasörler bulundu.")
print("Güvenlik kontrolleri başarılı.")
print("Kaynak klasörlerden hiçbir veri silinmeyecek.")


Kaynak klasörler bulundu.
Güvenlik kontrolleri başarılı.
Kaynak klasörlerden hiçbir veri silinmeyecek.


## 5. MediaPipe Face Detector Modelini İndirme


In [7]:
if not os.path.isfile(MODEL_PATH):
    print("Face Detector modeli indiriliyor...")
    urllib.request.urlretrieve(MODEL_URL, MODEL_PATH)

if not os.path.isfile(MODEL_PATH) or os.path.getsize(MODEL_PATH) == 0:
    raise RuntimeError("Face Detector model dosyası indirilemedi.")

print("Model hazır:", MODEL_PATH)
print("Model boyutu:", os.path.getsize(MODEL_PATH), "bayt")


Face Detector modeli indiriliyor...
Model hazır: /content/blaze_face_short_range.tflite
Model boyutu: 229746 bayt


## 6. Çıktı Klasörlerini Hazırlama


In [8]:
# Bu notebook temiz başlangıç yapacak şekilde ayarlanmıştır.
# Kaynak real_frames/fake_frames klasörlerine kesinlikle dokunulmaz.
CLEAN_OUTPUT_BEFORE_RUN = True

os.makedirs(OUTPUT_ROOT, exist_ok=True)

if CLEAN_OUTPUT_BEFORE_RUN:
    for class_name in ["Real", "Fake"]:
        class_output = os.path.join(OUTPUT_ROOT, class_name)

        if os.path.exists(class_output):
            shutil.rmtree(class_output)

        for split in ["train", "val", "test"]:
            os.makedirs(
                os.path.join(class_output, split),
                exist_ok=True
            )

    # Önceki çalıştırmadan kalan rapor/checkpoint dosyalarını temizle.
    for file_path in [
        METADATA_CSV_PATH,
        NO_FACE_LOG_PATH,
        CHECKPOINT_JSON_PATH,
        RUN_SUMMARY_PATH,
    ]:
        if os.path.isfile(file_path):
            os.remove(file_path)

    print("Eski çıktı sonuçları temizlendi ve yeni klasörler oluşturuldu.")
else:
    for class_name in ["Real", "Fake"]:
        for split in ["train", "val", "test"]:
            os.makedirs(
                os.path.join(OUTPUT_ROOT, class_name, split),
                exist_ok=True
            )
    print("Mevcut çıktı korunuyor; eksik klasörler oluşturuldu.")

print("Kaynak klasörlere dokunulmadı.")


Eski çıktı sonuçları temizlendi ve yeni klasörler oluşturuldu.
Kaynak klasörlere dokunulmadı.


## 7. Yardımcı Fonksiyonlar


In [9]:
def find_all_images(root_dir):
    """Bir klasörün altındaki desteklenen bütün görüntüleri döndürür."""
    paths = []

    for dirpath, _, filenames in os.walk(root_dir):
        for filename in filenames:
            if filename.lower().endswith(IMG_EXTENSIONS):
                paths.append(os.path.join(dirpath, filename))

    return sorted(paths)


def has_face(image_path, detector):
    """Görüntü okunabiliyor ve en az bir yüz içeriyorsa True döndürür."""
    image_bgr = cv2.imread(image_path)

    if image_bgr is None:
        return False, "goruntu_okunamadi"

    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    image_rgb = np.ascontiguousarray(image_rgb)

    mp_image = mp.Image(
        image_format=mp.ImageFormat.SRGB,
        data=image_rgb
    )

    detection_result = detector.detect(mp_image)

    if detection_result.detections:
        return True, "yuz_bulundu"

    return False, "yuz_bulunamadi"


def select_face_frames(
    frame_pool,
    target_count,
    detector,
    rejected_rows,
    class_name,
    split_name,
    seed
):
    """Havuzdan tam hedef sayıda yüzlü görüntü seçer."""
    candidates = frame_pool.copy()
    random.Random(seed).shuffle(candidates)

    selected = []
    split_rejected = 0

    progress = tqdm(
        candidates,
        desc=f"{class_name}/{split_name} yüz kontrolü"
    )

    for image_path in progress:
        if len(selected) >= target_count:
            break

        face_found, reason = has_face(image_path, detector)

        if face_found:
            selected.append(image_path)
        else:
            split_rejected += 1
            rejected_rows.append({
                "sinif": class_name,
                "split": split_name,
                "dosya": image_path,
                "neden": reason,
            })

        progress.set_postfix(
            secilen=len(selected),
            hedef=target_count,
            reddedilen=split_rejected
        )

    progress.close()

    if len(selected) < target_count:
        raise RuntimeError(
            f"{class_name}/{split_name} için yeterli yüzlü frame bulunamadı. "
            f"Hedef={target_count}, bulunan={len(selected)}, "
            f"toplam_havuz={len(frame_pool)}"
        )

    return selected


def create_unique_filename(class_name, split_name, index, src_path):
    extension = os.path.splitext(src_path)[1].lower()

    if extension not in IMG_EXTENSIONS:
        extension = ".jpg"

    return f"{class_name.lower()}_{split_name}_{index:05d}{extension}"


def copy_selected_frames(
    selected_paths,
    class_name,
    split_name,
    metadata_rows
):
    """Seçilen görüntüleri çıktı klasörüne kopyalar ve kaydı anında ekler."""
    destination_dir = os.path.join(
        OUTPUT_ROOT,
        class_name,
        split_name
    )

    os.makedirs(destination_dir, exist_ok=True)

    for index, src_path in enumerate(
        tqdm(selected_paths, desc=f"{class_name}/{split_name} kopyalama")
    ):
        filename = create_unique_filename(
            class_name,
            split_name,
            index,
            src_path
        )

        dst_path = os.path.join(destination_dir, filename)

        # copy2 kaynak görüntüyü silmez; yalnızca kopyalar.
        shutil.copy2(src_path, dst_path)

        metadata_rows.append({
            "sinif": class_name,
            "split": split_name,
            "orijinal_yol": src_path,
            "yeni_yol": dst_path,
            "dosya_adi": filename,
        })


def save_csv_files(metadata_rows, rejected_rows):
    """Metadata ve red logunu her split sonrasında diske yazar."""
    with open(
        METADATA_CSV_PATH,
        "w",
        newline="",
        encoding="utf-8-sig"
    ) as csv_file:
        writer = csv.DictWriter(
            csv_file,
            fieldnames=[
                "sinif",
                "split",
                "orijinal_yol",
                "yeni_yol",
                "dosya_adi",
            ]
        )
        writer.writeheader()
        writer.writerows(metadata_rows)

    with open(
        NO_FACE_LOG_PATH,
        "w",
        newline="",
        encoding="utf-8-sig"
    ) as csv_file:
        writer = csv.DictWriter(
            csv_file,
            fieldnames=[
                "sinif",
                "split",
                "dosya",
                "neden",
            ]
        )
        writer.writeheader()
        writer.writerows(rejected_rows)


def save_checkpoint(completed_splits, metadata_rows, rejected_rows):
    checkpoint = {
        "tamamlanan_splitler": completed_splits,
        "kopyalanan_frame_sayisi": len(metadata_rows),
        "reddedilen_frame_sayisi": len(rejected_rows),
    }

    with open(CHECKPOINT_JSON_PATH, "w", encoding="utf-8") as f:
        json.dump(checkpoint, f, ensure_ascii=False, indent=2)


def count_images(folder):
    return sum(
        1
        for filename in os.listdir(folder)
        if filename.lower().endswith(IMG_EXTENSIONS)
    )


## 8. Kaynak Görüntü Havuzlarını Toplama


In [10]:
source_pools = {
    "Real": {
        split: find_all_images(
            os.path.join(REAL_FRAMES_ROOT, split)
        )
        for split in ["train", "val", "test"]
    },
    "Fake": {
        split: find_all_images(
            os.path.join(FAKE_FRAMES_ROOT, split)
        )
        for split in ["train", "val", "test"]
    },
}

for class_name in ["Real", "Fake"]:
    print(f"\n{class_name} kaynak sayıları")

    for split_name in ["train", "val", "test"]:
        pool_count = len(source_pools[class_name][split_name])
        target_count = TARGETS_PER_SPLIT[split_name]

        print(
            f"  {split_name}: {pool_count} görüntü "
            f"| hedef: {target_count}"
        )

        if pool_count < target_count:
            raise RuntimeError(
                f"{class_name}/{split_name} klasöründe "
                f"hedef sayıdan daha az görüntü var."
            )



Real kaynak sayıları
  train: 21705 görüntü | hedef: 1200
  val: 2745 görüntü | hedef: 150
  test: 2742 görüntü | hedef: 150

Fake kaynak sayıları
  train: 135550 görüntü | hedef: 1200
  val: 16903 görüntü | hedef: 150
  test: 16932 görüntü | hedef: 150


## 9. Güncel MediaPipe Tasks API ile Yüzlü Frame Seçimi


In [11]:
rejected_rows = []
metadata_rows = []
selected_frames = {"Real": {}, "Fake": {}}
completed_splits = []

BaseOptions = mp.tasks.BaseOptions
FaceDetector = mp.tasks.vision.FaceDetector
FaceDetectorOptions = mp.tasks.vision.FaceDetectorOptions
RunningMode = mp.tasks.vision.RunningMode

detector_options = FaceDetectorOptions(
    base_options=BaseOptions(
        model_asset_path=MODEL_PATH
    ),
    running_mode=RunningMode.IMAGE,
    min_detection_confidence=FACE_MIN_CONFIDENCE
)

with FaceDetector.create_from_options(detector_options) as detector:

    for class_index, class_name in enumerate(["Real", "Fake"]):
        for split_index, split_name in enumerate(
            ["train", "val", "test"]
        ):
            seed = (
                RANDOM_SEED
                + class_index * 100
                + split_index
            )

            print("\n" + "=" * 70)
            print(f"İŞLENİYOR: {class_name}/{split_name}")
            print("=" * 70)

            selected = select_face_frames(
                frame_pool=source_pools[class_name][split_name],
                target_count=TARGETS_PER_SPLIT[split_name],
                detector=detector,
                rejected_rows=rejected_rows,
                class_name=class_name,
                split_name=split_name,
                seed=seed,
            )

            selected_frames[class_name][split_name] = selected

            # Kritik düzeltme:
            # Seçim bitince bütün sınıfları beklemeden bu split hemen kopyalanır.
            copy_selected_frames(
                selected_paths=selected,
                class_name=class_name,
                split_name=split_name,
                metadata_rows=metadata_rows,
            )

            completed_splits.append(f"{class_name}/{split_name}")

            # Metadata, log ve checkpoint her split sonunda anında yazılır.
            save_csv_files(metadata_rows, rejected_rows)
            save_checkpoint(
                completed_splits,
                metadata_rows,
                rejected_rows
            )

            copied_count = count_images(
                os.path.join(OUTPUT_ROOT, class_name, split_name)
            )

            expected_count = TARGETS_PER_SPLIT[split_name]

            if copied_count != expected_count:
                raise RuntimeError(
                    f"{class_name}/{split_name} kopyalama sayısı hatalı. "
                    f"Beklenen={expected_count}, bulunan={copied_count}"
                )

            print(
                f"{class_name}/{split_name} tamamlandı: "
                f"{copied_count} görüntü kopyalandı."
            )
            print("Metadata güncellendi:", METADATA_CSV_PATH)
            print("Checkpoint güncellendi:", CHECKPOINT_JSON_PATH)

print("\nBütün splitler başarıyla seçildi ve kopyalandı.")
print("Tamamlanan splitler:", completed_splits)
print("Toplam kopyalanan:", len(metadata_rows))
print("Toplam reddedilen:", len(rejected_rows))



İŞLENİYOR: Real/train


Real/train yüz kontrolü:   0%|          | 0/21705 [00:00<?, ?it/s]

Real/train kopyalama:   0%|          | 0/1200 [00:00<?, ?it/s]

Real/train tamamlandı: 1200 görüntü kopyalandı.
Metadata güncellendi: /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Deney 1 Frame/secim_metadata.csv
Checkpoint güncellendi: /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Deney 1 Frame/islem_checkpoint.json

İŞLENİYOR: Real/val


Real/val yüz kontrolü:   0%|          | 0/2745 [00:00<?, ?it/s]

...

Fake/val kopyalama:   0%|          | 0/150 [00:00<?, ?it/s]

Fake/val tamamlandı: 150 görüntü kopyalandı.
Metadata güncellendi: /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Deney 1 Frame/secim_metadata.csv
Checkpoint güncellendi: /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Deney 1 Frame/islem_checkpoint.json

İŞLENİYOR: Fake/test


Fake/test yüz kontrolü:   0%|          | 0/16932 [00:00<?, ?it/s]

Fake/test kopyalama:   0%|          | 0/150 [00:00<?, ?it/s]

Fake/test tamamlandı: 150 görüntü kopyalandı.
Metadata güncellendi: /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Deney 1 Frame/secim_metadata.csv
Checkpoint güncellendi: /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Deney 1 Frame/islem_checkpoint.json

Bütün splitler başarıyla seçildi ve kopyalandı.
Tamamlanan splitler: ['Real/train', 'Real/val', 'Real/test', 'Fake/train', 'Fake/val', 'Fake/test']
Toplam kopyalanan: 3000
Toplam reddedilen: 206


## 10. Seçilen Görüntüleri Kopyalama


In [12]:
# Ana işlem hücresinde her split seçildikten hemen sonra kopyalandı.
# Bu hücre yalnızca ara sonucu gösterir.

print("Tamamlanan splitler:", completed_splits)
print("Metadata satır sayısı:", len(metadata_rows))
print("Reddedilen satır sayısı:", len(rejected_rows))

for class_name in ["Real", "Fake"]:
    for split_name in ["train", "val", "test"]:
        folder = os.path.join(
            OUTPUT_ROOT,
            class_name,
            split_name
        )
        print(
            f"{class_name}/{split_name}: "
            f"{count_images(folder)} görüntü"
        )


Tamamlanan splitler: ['Real/train', 'Real/val', 'Real/test', 'Fake/train', 'Fake/val', 'Fake/test']
Metadata satır sayısı: 3000
Reddedilen satır sayısı: 206
Real/train: 1200 görüntü
Real/val: 150 görüntü
Real/test: 150 görüntü
Fake/train: 1200 görüntü
Fake/val: 150 görüntü
Fake/test: 150 görüntü


## 11. Metadata ve Log Dosyalarını Kaydetme


In [13]:
# Dosyalar ana işlem sırasında her split sonrasında kaydedildi.
# Burada son kez tekrar güvenli biçimde yazılır.

save_csv_files(metadata_rows, rejected_rows)
save_checkpoint(completed_splits, metadata_rows, rejected_rows)

print("Metadata oluşturuldu:", METADATA_CSV_PATH)
print("Reddedilen frame logu oluşturuldu:", NO_FACE_LOG_PATH)
print("Checkpoint oluşturuldu:", CHECKPOINT_JSON_PATH)

assert os.path.isfile(METADATA_CSV_PATH), "Metadata CSV oluşturulamadı."
assert os.path.isfile(NO_FACE_LOG_PATH), "Red logu oluşturulamadı."
assert os.path.isfile(CHECKPOINT_JSON_PATH), "Checkpoint oluşturulamadı."

print("Bütün kayıt dosyaları doğrulandı.")


Metadata oluşturuldu: /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Deney 1 Frame/secim_metadata.csv
Reddedilen frame logu oluşturuldu: /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Deney 1 Frame/yuzsuz_veya_okunamayan_frameler.csv
Checkpoint oluşturuldu: /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Deney 1 Frame/islem_checkpoint.json
Bütün kayıt dosyaları doğrulandı.


## 12. Kesin Sonuç Doğrulaması


In [14]:
actual_total = 0

print("=" * 70)
print("ÇIKTI DOĞRULAMA")
print("=" * 70)

for class_name in ["Real", "Fake"]:
    class_total = 0

    for split_name in ["train", "val", "test"]:
        folder = os.path.join(
            OUTPUT_ROOT,
            class_name,
            split_name
        )

        actual = count_images(folder)
        expected = TARGETS_PER_SPLIT[split_name]

        print(
            f"{class_name:4s}/{split_name:5s}: "
            f"{actual:4d} | beklenen: {expected:4d}"
        )

        assert actual == expected, (
            f"{class_name}/{split_name} sayısı yanlış. "
            f"Bulunan={actual}, beklenen={expected}"
        )

        class_total += actual
        actual_total += actual

    assert class_total == TARGET_PER_CLASS
    print(f"{class_name} toplam: {class_total}\n")

assert actual_total == TOTAL_TARGET
assert len(metadata_rows) == TOTAL_TARGET
assert os.path.isfile(METADATA_CSV_PATH)

# Aynı tam kaynak yolunun iki kere seçilmediğini doğrula.
source_paths = [
    row["orijinal_yol"]
    for row in metadata_rows
]
assert len(source_paths) == len(set(source_paths)), (
    "Aynı kaynak görüntü birden fazla kez seçilmiş."
)

# Her sınıfta train/val/test kaynak yolları birbirinden ayrıdır.
for class_name in ["Real", "Fake"]:
    split_sets = {
        split: {
            row["orijinal_yol"]
            for row in metadata_rows
            if row["sinif"] == class_name and row["split"] == split
        }
        for split in ["train", "val", "test"]
    }

    assert not (split_sets["train"] & split_sets["val"])
    assert not (split_sets["train"] & split_sets["test"])
    assert not (split_sets["val"] & split_sets["test"])

print(f"GENEL TOPLAM: {actual_total}")
print("Dağılım: 1500 Real + 1500 Fake")
print("Her sınıf: 1200 train + 150 val + 150 test")
print("Train/val/test arasında aynı kaynak yolu bulunmadı.")
print("Metadata CSV başarıyla oluşturuldu.")
print("Kaynak görüntüler silinmedi; yalnızca kopyalandı.")


ÇIKTI DOĞRULAMA
Real/train: 1200 | beklenen: 1200
Real/val  :  150 | beklenen:  150
Real/test :  150 | beklenen:  150
Real toplam: 1500

Fake/train: 1200 | beklenen: 1200
Fake/val  :  150 | beklenen:  150
Fake/test :  150 | beklenen:  150
Fake toplam: 1500

GENEL TOPLAM: 3000
Dağılım: 1500 Real + 1500 Fake
Her sınıf: 1200 train + 150 val + 150 test
Train/val/test arasında aynı kaynak yolu bulunmadı.
Metadata CSV başarıyla oluşturuldu.
Kaynak görüntüler silinmedi; yalnızca kopyalandı.


## 13. İşlem Özeti


In [15]:
summary = {
    "genel_toplam": len(metadata_rows),
    "real": {
        split: count_images(
            os.path.join(OUTPUT_ROOT, "Real", split)
        )
        for split in ["train", "val", "test"]
    },
    "fake": {
        split: count_images(
            os.path.join(OUTPUT_ROOT, "Fake", split)
        )
        for split in ["train", "val", "test"]
    },
    "reddedilen_frame_sayisi": len(rejected_rows),
    "tamamlanan_splitler": completed_splits,
    "metadata_csv": METADATA_CSV_PATH,
    "output_root": OUTPUT_ROOT,
}

with open(RUN_SUMMARY_PATH, "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print(json.dumps(summary, ensure_ascii=False, indent=2))
print("İşlem özeti kaydedildi:", RUN_SUMMARY_PATH)


{
  "genel_toplam": 3000,
  "real": {
    "train": 1200,
    "val": 150,
    "test": 150
  },
  "fake": {
    "train": 1200,
    "val": 150,
    "test": 150
  },
  "reddedilen_frame_sayisi": 206,
  "tamamlanan_splitler": [
    "Real/train",
    "Real/val",
    "Real/test",
    "Fake/train",
    "Fake/val",
    "Fake/test"
  ],
  "metadata_csv": "/content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Deney 1 Frame/secim_metadata.csv",
  "output_root": "/content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Deney 1 Frame"
}
İşlem özeti kaydedildi: /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Deney 1 Frame/islem_ozeti.json
